# ANALISIS DE DEPENDENCIAS UNIVERSALES

In [11]:
import json
import statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import spacy
from spacy.tokens import Doc
from tqdm import tqdm

nlp = spacy.load("es_core_news_sm")

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F8',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# ---- Rutas — ajusta si es necesario ----
BASE = Path().resolve().parent   # sube de baselines/ → raíz
DATA = BASE

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def safe_str(v):
    return '+'.join(sorted(str(x) for x in v)) if isinstance(v, list) else str(v)

mt = load_jsonl(DATA / 'menu_train.jsonl')
md = load_jsonl(DATA / 'menu_dev.jsonl')
rt = load_jsonl(DATA / 'recipe_train.jsonl')
rd = load_jsonl(DATA / 'recipe_dev.jsonl')


In [12]:
nlp = spacy.load("es_core_news_sm")
def parse_tokens(tokens: list[str]) -> Doc:
    doc = Doc(nlp.vocab, words=tokens)
    for name, proc in nlp.pipeline:
        if name != "tok2vec":
            try:
                doc = proc(doc)
            except Exception:
                pass
    return doc

In [13]:
def build_menu_df(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        base = {
            "id"                  : rec["id"],
            "task_id"             : rec["task_id"],
            "annotator_id"        : rec["annotator_id"],
            "tokens"              : rec["tokens"],
            "ner_tags"            : rec["ner_tags"],
            "meta_origin_cuisine" : rec["meta_origin_cuisine"],
            "meta_language"       : rec["meta_language"],
        }
        for ent in rec["entities"]:
            rows.append({
                **base,
                "ent_text"           : ent["text"],
                "ent_label"          : ent["label"],
                "token_start"        : ent["token_start"],
                "token_end"          : ent["token_end"],
                "course_type"        : ent.get("course_type"),        # solo en DISH
                "is_first_occurrence": ent.get("is_first_occurrence"),
                "attrs_propagated"   : ent.get("attrs_propagated"),
            })
    return pd.DataFrame(rows)

In [14]:
def build_token_df(records):
    rows = []
    for rec in records:
        tokens   = rec["tokens"]
        ner_tags = rec["ner_tags"]
        
        # Mapa: índice de token → metadata de su entidad
        token_to_ent = {}
        for ent in rec["entities"]:
            for i in range(ent["token_start"], ent["token_end"] + 1):
                token_to_ent[i] = ent
        
        doc = parse_tokens(tokens)
        
        for i, token in enumerate(doc):
            ent = token_to_ent.get(i, {})
            rows.append({
                # token base
                "text"               : token.text,
                "ner_tag"            : ner_tags[i],
                # features UD
                "pos"                : token.pos_,
                "dep"                : token.dep_,
                "head_pos"           : token.head.pos_,
                "head_dep"           : token.head.dep_,
                "is_root"            : token.dep_ == "ROOT",
                "dist_head"          : token.i - token.head.i,
                "n_children"         : len(list(token.children)),
                "prev_pos"           : doc[i-1].pos_ if i > 0 else "<START>",
                "next_pos"           : doc[i+1].pos_ if i < len(doc)-1 else "<END>",
                "prev_dep"           : doc[i-1].dep_ if i > 0 else "<START>",
                "next_dep"           : doc[i+1].dep_ if i < len(doc)-1 else "<END>",
                # metadata de entidad (None si token es O)
                "course_type"        : ent.get("course_type"),
                "is_first_occurrence": ent.get("is_first_occurrence"),
                "attrs_propagated"   : ent.get("attrs_propagated"),
            })
    return pd.DataFrame(rows)

In [15]:
print(f'menu_train : {len(mt):,} seqs')
print(f'menu_dev   : {len(md):,} seqs')
print(f'recipe_train: {len(rt):,} seqs')
print(f'recipe_dev  : {len(rd):,} seqs')

menu = build_menu_df(mt)

menu_train : 11,070 seqs
menu_dev   : 2,366 seqs
recipe_train: 3,358 seqs
recipe_dev  : 719 seqs


In [16]:

menu.head(5)

,id,task_id,annotator_id,tokens,ner_tags,meta_origin_cuisine,meta_language,ent_text,ent_label,token_start,token_end,course_type,is_first_occurrence,attrs_propagated
0,1250,40769,47,"[torta, de, arándanos]","[B-DISH, I-DISH, I-DISH]",Other,Spanish,torta de arándanos,DISH,0,2,dessert,True,False
1,1283,40798,47,"[carajillo, espresso, licor, 43]","[B-BEVERAGE, B-BEVERAGE, B-BEVERAGE, I-BEVERAGE]",Spanish,Spanish,carajillo,BEVERAGE,0,0,None,True,False
2,1283,40798,47,"[carajillo, espresso, licor, 43]","[B-BEVERAGE, B-BEVERAGE, B-BEVERAGE, I-BEVERAGE]",Spanish,Spanish,espresso,BEVERAGE,1,1,None,True,False
3,1283,40798,47,"[carajillo, espresso, licor, 43]","[B-BEVERAGE, B-BEVERAGE, B-BEVERAGE, I-BEVERAGE]",Spanish,Spanish,licor 43,BEVERAGE,2,3,None,True,False
4,35,11011,31,"[iwai, tradition]","[B-BEVERAGE, I-BEVERAGE]",Brazilian,Spanish,iwai tradition,BEVERAGE,0,1,None,True,False


In [17]:
token_df = build_token_df(mt)  

In [19]:
palabras = {}
for row in tqdm(token_df['text']):
    doc = nlp(row.lower())
    for token in doc:
        if token.is_alpha and not token.is_stop:
            if token.text in palabras:
                num_temp = int(palabras[token.text])
                palabras[token.text] = num_temp + 1
            else:
                palabras[token.text] = 1
# Guardar en archivo de texto, ordenado por frecuencia (mayor a menor)
with open('palabras.txt', 'w', encoding='utf-8') as f:
    for palabra, frecuencia in sorted(palabras.items(), key=lambda x: x[1], reverse=True):
        f.write(f"{palabra}: {frecuencia}\n")

print("Archivo guardado exitosamente.")

  0%|          | 0/72579 [00:00<?, ?it/s]

100%|██████████| 72579/72579 [10:59<00:00, 110.04it/s]


Archivo guardado exitosamente.


In [20]:
pos_freq = {}
for row in tqdm(menu['ent_text'].to_list()):
    doc = nlp(row.lower())
    for token in doc:
        if token.pos_ in pos_freq:
            value = pos_freq[token.pos_]
            pos_freq[token.pos_] = value + 1
        else:
            pos_freq[token.pos_] =  1
# Guardar en archivo de texto, ordenado por frecuencia (mayor a menor)
with open('pos.txt', 'w', encoding='utf-8') as f:
    for pos, frecuencia in sorted(pos_freq.items(), key=lambda x: x[1], reverse=True):
        f.write(f"{pos}: {frecuencia}\n")

print("Archivo guardado exitosamente.")

  0%|          | 0/33871 [00:00<?, ?it/s]

100%|██████████| 33871/33871 [05:53<00:00, 95.88it/s] 

Archivo guardado exitosamente.


In [21]:
df_pos = pd.DataFrame([[key, pos_freq[key]] for key in pos_freq.keys()], columns=['POS', 'Freq'])
df_pos.sort_values('Freq').tail(15)
df_pos = df_pos[:20]
df_pos.head(10)

,POS,Freq
0,VERB,8729
1,ADP,7571
2,NOUN,26857
3,PROPN,10862
4,NUM,1537
5,ADV,378
6,ADJ,14190
7,PUNCT,450
8,CCONJ,611
9,DET,1194


In [24]:
B_DISH_df = token_df[token_df['ner_tag'] == 'B-DISH']
I_DISH_df = token_df[token_df['ner_tag'] == 'I-DISH']

B_BEVERAGE_df = token_df[token_df['ner_tag'] == 'B-BEVERAGE']
I_BEVERAGE_df = token_df[token_df['ner_tag'] == 'I-BEVERAGE']

B_INGREDIENT_df = token_df[token_df['ner_tag'] == 'B-INGREDIENT']
I_INGREDIENT_df = token_df[token_df['ner_tag'] == 'I-INGREDIENT']

B_BRAND_df = token_df[token_df['ner_tag'] == 'B-BRAND']
I_BRAND_df = token_df[token_df['ner_tag'] == 'I-BRAND']

In [27]:
B_DISH_df.head(10)

,text,ner_tag,pos,dep,head_pos,head_dep,is_root,dist_head,n_children,prev_pos,next_pos,prev_dep,next_dep,course_type,is_first_occurrence,attrs_propagated
0,torta,B-DISH,NOUN,det,NOUN,ROOT,False,-1,0,<START>,NOUN,<START>,ROOT,dessert,True,False
13,alitas,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,main_course,True,False
29,wings,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,main_course,True,False
54,calientito,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,main_course,True,False
66,choclo,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,main_course,True,False
73,ensalada,B-DISH,NOUN,det,NOUN,ROOT,False,-1,0,<START>,NOUN,<START>,ROOT,appetizer,True,False
89,dedos,B-DISH,NOUN,ROOT,NOUN,ROOT,True,0,1,<START>,NOUN,<START>,punct,appetizer,True,False
97,causa,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,main_course,True,False
110,ceviche,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,appetizer,True,False
149,cheese,B-DISH,NOUN,det,NOUN,det,False,-1,0,<START>,NOUN,<START>,det,dessert,True,False


In [33]:
for col in B_DISH_df.columns:
    print(f"\n{col}")
    print(B_DISH_df[col].value_counts())


text
text
ensalada      125
papas         119
arroz          99
ceviche        88
pollo          74
             ... 
hamburgues      1
goycochea       1
chistorra       1
sandwiches      1
kamaron         1
Name: count, Length: 1803, dtype: int64

ner_tag
ner_tag
B-DISH    5430
Name: count, dtype: int64

pos
pos
NOUN    5430
Name: count, dtype: int64

dep
dep
det      4676
ROOT      696
punct      58
Name: count, dtype: int64

head_pos
head_pos
NOUN    5430
Name: count, dtype: int64

head_dep
head_dep
det     4143
ROOT    1287
Name: count, dtype: int64

is_root
is_root
False    4734
True      696
Name: count, dtype: int64

dist_head
dist_head
-1    4676
 0     696
 1      58
Name: count, dtype: int64

n_children
n_children
0    4552
1     792
2      86
Name: count, dtype: int64

prev_pos
prev_pos
<START>    4867
NOUN        563
Name: count, dtype: int64

next_pos
next_pos
NOUN     5135
<END>     295
Name: count, dtype: int64

prev_dep
prev_dep
<START>    4867
det         505
ROOT    